# EXP13: Keltner 通道参数扫描 (EMA周期 + 通道倍数)

默认 KC_EMA=20, KC_MULT=1.5。测试不同组合对突破信号质量的影响。

KC_EMA [15,20,25,30] x KC_MULT [1.2,1.5,1.8,2.0] = 16 变体

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Ready')

In [ ]:
results = []
for kc_ema in [15, 20, 25, 30]:
    for kc_mult in [1.2, 1.5, 1.8, 2.0]:
        data = DataBundle.load_custom(kc_ema=kc_ema, kc_mult=kc_mult)
        r = run_variant(data, f"EMA{kc_ema}_M{kc_mult}", **LIVE_KWARGS)
        r['kc_ema'] = kc_ema
        r['kc_mult'] = kc_mult
        results.append(r)
        print(f"  EMA{kc_ema}_M{kc_mult}: Sharpe={r['sharpe']:.2f}, N={r['n']}, PnL=${r['total_pnl']:.0f}")

print_ranked(results)

In [ ]:
df = pd.DataFrame([{'kc_ema': r['kc_ema'], 'kc_mult': r['kc_mult'],
                     'sharpe': r['sharpe'], 'total_pnl': r['total_pnl'], 'n': r['n']} for r in results])

print("\nSharpe Heatmap (rows=KC_EMA, cols=KC_MULT):")
print(df.pivot_table(index='kc_ema', columns='kc_mult', values='sharpe').round(2))

print("\nTrade Count Heatmap:")
print(df.pivot_table(index='kc_ema', columns='kc_mult', values='n'))

print("\nPnL Heatmap:")
print(df.pivot_table(index='kc_ema', columns='kc_mult', values='total_pnl').round(0))

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp13_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')